In [1]:
from pynq import Overlay
overlay = Overlay("mmult_accel.bit")
print(overlay.ip_dict.keys())

dict_keys(['mmult_accel_0', 'zynq_ultra_ps_e_0'])


In [2]:
mmult_accel = overlay.mmult_accel_0
print(mmult_accel.register_map)

RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  A_1 = Register(A=write-only),
  A_2 = Register(A=write-only),
  B_1 = Register(B=write-only),
  B_2 = Register(B=write-only),
  C_1 = Register(C=write-only),
  C_2 = Register(C=write-only),
  N = Register(N=write-only),
  K = Register(K=write-only),
  M = Register(M=write-only),
  update_A = Register(update_A=write-only)
}


In [3]:
import numpy as np
import torch
import time
from pynq import allocate

def set_ptr(reg1, reg2, addr):
    setattr(mmult_accel.register_map, reg1, addr & 0xFFFFFFFF)
    setattr(mmult_accel.register_map, reg2, (addr >> 32) & 0xFFFFFFFF)

def benchmark_gemm(N, K, M, label=""):
    A_np = np.random.randint(-5, 5, size=(N, K)).astype(np.int8)
    B_np = np.random.randint(-5, 5, size=(K, M)).astype(np.int8)
    flops = 2 * N * K * M

    t0 = time.time()
    C_numpy = A_np.astype(np.int32) @ B_np.astype(np.int32)
    t_numpy = time.time() - t0

    A_t = torch.from_numpy(A_np.astype(np.int32))
    B_t = torch.from_numpy(B_np.astype(np.int32))
    t0 = time.time()
    C_torch = A_t @ B_t
    t_torch = time.time() - t0

    A = allocate(shape=(N, K), dtype=np.int8)
    B = allocate(shape=(K, M), dtype=np.int8)
    C = allocate(shape=(N, M), dtype=np.int32)
    A[:] = A_np; B[:] = B_np; C[:] = 0

    t0 = time.time()
    A.flush(); B.flush()
    set_ptr('A_1','A_2', A.physical_address)
    set_ptr('B_1','B_2', B.physical_address)
    set_ptr('C_1','C_2', C.physical_address)
    mmult_accel.register_map.N = N
    mmult_accel.register_map.K = K
    mmult_accel.register_map.M = M
    mmult_accel.register_map.update_A = 1
    t_compute_start = time.time()
    mmult_accel.register_map.CTRL.AP_START = 1
    while mmult_accel.register_map.CTRL.AP_DONE == 0:
        pass
    t_compute = time.time() - t_compute_start
    C.invalidate()
    t_e2e = time.time() - t0

    correct = np.array_equal(np.array(C), C_numpy)

    print(f"--- {label} (N={N}, K={K}, M={M}) Dung={correct} ---")
    print(f"  NumPy         : {t_numpy*1000:10.3f} ms  {flops/t_numpy/1e9:8.4f} GFLOPs")
    print(f"  PyTorch       : {t_torch*1000:10.3f} ms  {flops/t_torch/1e9:8.4f} GFLOPs")
    print(f"  FPGA(compute) : {t_compute*1000:10.3f} ms  {flops/t_compute/1e9:8.4f} GFLOPs")
    print(f"  FPGA(e2e)     : {t_e2e*1000:10.3f} ms  {flops/t_e2e/1e9:8.4f} GFLOPs")
    return dict(N=N, K=K, M=M, t_numpy=t_numpy, t_torch=t_torch, t_compute=t_compute, t_e2e=t_e2e, correct=correct)

print("=== Case chuan bai bao ===")
r1 = benchmark_gemm(64, 768, 768, "Attention case")
print()
r2 = benchmark_gemm(64, 768, 3072, "FFN case")

=== Case chuan bai bao ===
--- Attention case (N=64, K=768, M=768) Dung=True ---
  NumPy         :   1271.195 ms    0.0594 GFLOPs
  PyTorch       :     99.738 ms    0.7570 GFLOPs
  FPGA(compute) :     16.346 ms    4.6188 GFLOPs
  FPGA(e2e)     :     17.045 ms    4.4294 GFLOPs

--- FFN case (N=64, K=768, M=3072) Dung=True ---
  NumPy         :  21781.541 ms    0.0139 GFLOPs
  PyTorch       :    331.622 ms    0.9106 GFLOPs
  FPGA(compute) :     64.147 ms    4.7078 GFLOPs
  FPGA(e2e)     :     64.854 ms    4.6565 GFLOPs


In [4]:
print("=== Quet N (K=M=768 co dinh) ===")
sweep_results = []
for N_test in [8, 16, 32, 64]:
    r = benchmark_gemm(N_test, 768, 768, f"N={N_test}")
    sweep_results.append(r)
    print()

print(f"{'N':>6}{'FPGA compute(ms)':>20}{'GFLOPs':>12}{'PyTorch(ms)':>15}{'Speedup vs PyTorch':>20}")
for r in sweep_results:
    flops = 2*r['N']*r['K']*r['M']
    gflops = flops/r['t_compute']/1e9
    speedup = r['t_torch']/r['t_compute']
    print(f"{r['N']:>6}{r['t_compute']*1000:>18.3f}{gflops:>12.3f}{r['t_torch']*1000:>13.3f}{speedup:>18.2f}x")

=== Quet N (K=M=768 co dinh) ===
--- N=8 (N=8, K=768, M=768) Dung=True ---
  NumPy         :    174.716 ms    0.0540 GFLOPs
  PyTorch       :     11.487 ms    0.8216 GFLOPs
  FPGA(compute) :      9.410 ms    1.0028 GFLOPs
  FPGA(e2e)     :     10.020 ms    0.9419 GFLOPs

--- N=16 (N=16, K=768, M=768) Dung=True ---
  NumPy         :    335.063 ms    0.0563 GFLOPs
  PyTorch       :     21.940 ms    0.8603 GFLOPs
  FPGA(compute) :      9.830 ms    1.9202 GFLOPs
  FPGA(e2e)     :     10.547 ms    1.7895 GFLOPs

--- N=32 (N=32, K=768, M=768) Dung=True ---
  NumPy         :    660.905 ms    0.0571 GFLOPs
  PyTorch       :     41.998 ms    0.8988 GFLOPs
  FPGA(compute) :     10.555 ms    3.5764 GFLOPs
  FPGA(e2e)     :     11.123 ms    3.3936 GFLOPs

--- N=64 (N=64, K=768, M=768) Dung=True ---
  NumPy         :   1269.027 ms    0.0595 GFLOPs
  PyTorch       :     82.822 ms    0.9116 GFLOPs
  FPGA(compute) :     16.342 ms    4.6199 GFLOPs
  FPGA(e2e)     :     16.922 ms    4.4615 GFLOPs

     

In [5]:
with open('/sys/class/hwmon/hwmon2/power1_input') as f:
    print("Cong suat hien tai:", int(f.read().strip())/1_000_000, "W")
    

Cong suat hien tai: 3.85 W


In [6]:
import time
def read_power_watts():
    with open('/sys/class/hwmon/hwmon2/power1_input') as f:
        return int(f.read().strip()) / 1_000_000

idle_samples = [read_power_watts() for _ in range(20)]
time.sleep(0.05)
idle_power = sum(idle_samples) / len(idle_samples)
print(f"Cong suat idle (baseline): {idle_power:.3f} W")

Cong suat idle (baseline): 3.785 W


In [7]:
import threading

def measure_power_during(func, *args, sample_interval=0.05, **kwargs):
    samples = []
    stop_flag = threading.Event()

    def sampler():
        while not stop_flag.is_set():
            samples.append(read_power_watts())
            time.sleep(sample_interval)

    t = threading.Thread(target=sampler)
    t.start()
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    stop_flag.set()
    t.join()

    avg_power = sum(samples) / len(samples) if samples else read_power_watts()
    energy_j = avg_power * elapsed
    return result, avg_power, energy_j, elapsed, len(samples)

N, K, M = 64, 768, 3072
A_np = np.random.randint(-5, 5, size=(N, K)).astype(np.int8)
B_np = np.random.randint(-5, 5, size=(K, M)).astype(np.int8)

def cpu_matmul():
    return A_np.astype(np.int32) @ B_np.astype(np.int32)

_, power_cpu, energy_cpu, t_cpu, n_cpu = measure_power_during(cpu_matmul)
print(f"CPU  : cong suat TB={power_cpu:.3f}W  thoi gian={t_cpu*1000:.1f}ms  nang luong={energy_cpu*1000:.2f}mJ  (so mau={n_cpu})")

A_t = torch.from_numpy(A_np.astype(np.int32))
B_t = torch.from_numpy(B_np.astype(np.int32))
def pytorch_matmul():
    return A_t @ B_t

_, power_pytorch, energy_pytorch, t_pytorch, n_pytorch = measure_power_during(pytorch_matmul)
print(f"PyTorch : cong suat TB={power_pytorch:.3f}W  thoi gian={t_pytorch*1000:.1f}ms  nang luong={energy_pytorch*1000:.2f}mJ  (so mau={n_pytorch})")

def fpga_matmul_once():
    A = allocate(shape=(N, K), dtype=np.int8)
    B = allocate(shape=(K, M), dtype=np.int8)
    C = allocate(shape=(N, M), dtype=np.int32)
    A[:] = A_np; B[:] = B_np; C[:] = 0
    A.flush(); B.flush()
    set_ptr('A_1','A_2', A.physical_address)
    set_ptr('B_1','B_2', B.physical_address)
    set_ptr('C_1','C_2', C.physical_address)
    mmult_accel.register_map.N = N
    mmult_accel.register_map.K = K
    mmult_accel.register_map.M = M
    mmult_accel.register_map.update_A = 1
    mmult_accel.register_map.CTRL.AP_START = 1
    while mmult_accel.register_map.CTRL.AP_DONE == 0:
        pass
    C.invalidate()

def fpga_matmul_repeated(iterations=20):
    for _ in range(iterations):
        fpga_matmul_once()

ITER = 20
_, power_fpga, energy_fpga_total, t_fpga_total, n_fpga = measure_power_during(fpga_matmul_repeated, iterations=ITER)
energy_fpga = energy_fpga_total / ITER
t_fpga = t_fpga_total / ITER
print(f"FPGA : cong suat TB={power_fpga:.3f}W  thoi gian/lan={t_fpga*1000:.1f}ms  nang luong/lan={energy_fpga*1000:.2f}mJ  (so mau={n_fpga}, lap {ITER} lan)")

print()
print(f"Baseline idle: {idle_power:.3f}W")
if energy_fpga > 0:
    print(f"Hieu suat nang luong FPGA vs PyTorch: {energy_pytorch/energy_fpga:.2f}x tot hon   [bai bao: ~4x]")

CPU  : cong suat TB=3.723W  thoi gian=22104.3ms  nang luong=82285.02mJ  (so mau=437)
PyTorch : cong suat TB=3.751W  thoi gian=331.9ms  nang luong=1244.92mJ  (so mau=7)
FPGA : cong suat TB=3.881W  thoi gian/lan=71.5ms  nang luong/lan=277.55mJ  (so mau=21, lap 20 lan)

Baseline idle: 3.785W
Hieu suat nang luong FPGA vs PyTorch: 4.49x tot hon   [bai bao: ~4x]


In [8]:
from transformers import AutoTokenizer, AutoModel

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [9]:
class FPGAQuantizedLinear(nn.Module):
    def __init__(self, linear, fpga_ip):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.fpga = fpga_ip
        W = linear.weight.detach().numpy()
        self.bias = linear.bias.detach().numpy().astype(np.float32) if linear.bias is not None else None
        w_scale = np.abs(W).max() / 127.0
        self.w_scale = w_scale if w_scale > 0 else 1e-8
        W_int8 = np.round(W / self.w_scale).astype(np.int8)
        self.B_int8 = np.ascontiguousarray(W_int8.T)

    def forward(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        X_int8 = np.round(x2d / x_scale).astype(np.int8)

        A = allocate(shape=(N, K), dtype=np.int8)
        B = allocate(shape=(K, M), dtype=np.int8)
        C = allocate(shape=(N, M), dtype=np.int32)
        A[:] = X_int8; B[:] = self.B_int8; C[:] = 0
        A.flush(); B.flush()
        set_ptr('A_1','A_2', A.physical_address)
        set_ptr('B_1','B_2', B.physical_address)
        set_ptr('C_1','C_2', C.physical_address)
        self.fpga.register_map.N = N
        self.fpga.register_map.K = K
        self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        C.invalidate()
        result = np.array(C).astype(np.float32) * (x_scale * self.w_scale)
        if self.bias is not None:
            result = result + self.bias
        return torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features)

import torch.nn as nn
sample_input = torch.randn(32, 768) * 0.5
layer0 = model.transformer.layer[0]
original_q_lin = layer0.attention.q_lin
fpga_q_lin = FPGAQuantizedLinear(original_q_lin, mmult_accel)

with torch.no_grad():
    out_cpu = original_q_lin(sample_input)
    out_fpga = fpga_q_lin(sample_input)

diff = (out_cpu - out_fpga).abs()
rel_error_norm = (out_cpu - out_fpga).norm() / out_cpu.norm() * 100
print("CPU output (5 gia tri dau):", out_cpu[0,:5])
print("FPGA output (5 gia tri dau):", out_fpga[0,:5])
print(f"Sai so tuong doi (norm): {rel_error_norm.item():.3f}%")

NameError: name 'nn' is not defined

In [10]:
class FPGAQuantizedLinear(nn.Module):
    def __init__(self, linear, fpga_ip):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.fpga = fpga_ip
        W = linear.weight.detach().numpy()
        self.bias = linear.bias.detach().numpy().astype(np.float32) if linear.bias is not None else None
        w_scale = np.abs(W).max() / 127.0
        self.w_scale = w_scale if w_scale > 0 else 1e-8
        W_int8 = np.round(W / self.w_scale).astype(np.int8)
        self.B_int8 = np.ascontiguousarray(W_int8.T)

    def forward(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        X_int8 = np.round(x2d / x_scale).astype(np.int8)

        A = allocate(shape=(N, K), dtype=np.int8)
        B = allocate(shape=(K, M), dtype=np.int8)
        C = allocate(shape=(N, M), dtype=np.int32)
        A[:] = X_int8; B[:] = self.B_int8; C[:] = 0
        A.flush(); B.flush()
        set_ptr('A_1','A_2', A.physical_address)
        set_ptr('B_1','B_2', B.physical_address)
        set_ptr('C_1','C_2', C.physical_address)
        self.fpga.register_map.N = N
        self.fpga.register_map.K = K
        self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        C.invalidate()
        result = np.array(C).astype(np.float32) * (x_scale * self.w_scale)
        if self.bias is not None:
            result = result + self.bias
        return torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features)

import torch.nn as nn
sample_input = torch.randn(32, 768) * 0.5
layer0 = model.transformer.layer[0]
original_q_lin = layer0.attention.q_lin
fpga_q_lin = FPGAQuantizedLinear(original_q_lin, mmult_accel)

with torch.no_grad():
    out_cpu = original_q_lin(sample_input)
    out_fpga = fpga_q_lin(sample_input)

diff = (out_cpu - out_fpga).abs()
rel_error_norm = (out_cpu - out_fpga).norm() / out_cpu.norm() * 100
print("CPU output (5 gia tri dau):", out_cpu[0,:5])
print("FPGA output (5 gia tri dau):", out_fpga[0,:5])
print(f"Sai so tuong doi (norm): {rel_error_norm.item():.3f}%")

NameError: name 'nn' is not defined

In [11]:
import torch.nn as nn

class FPGAQuantizedLinear(nn.Module):
    def __init__(self, linear, fpga_ip):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.fpga = fpga_ip
        W = linear.weight.detach().numpy()
        self.bias = linear.bias.detach().numpy().astype(np.float32) if linear.bias is not None else None
        w_scale = np.abs(W).max() / 127.0
        self.w_scale = w_scale if w_scale > 0 else 1e-8
        W_int8 = np.round(W / self.w_scale).astype(np.int8)
        self.B_int8 = np.ascontiguousarray(W_int8.T)

    def forward(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        X_int8 = np.round(x2d / x_scale).astype(np.int8)

        A = allocate(shape=(N, K), dtype=np.int8)
        B = allocate(shape=(K, M), dtype=np.int8)
        C = allocate(shape=(N, M), dtype=np.int32)
        A[:] = X_int8; B[:] = self.B_int8; C[:] = 0
        A.flush(); B.flush()
        set_ptr('A_1','A_2', A.physical_address)
        set_ptr('B_1','B_2', B.physical_address)
        set_ptr('C_1','C_2', C.physical_address)
        self.fpga.register_map.N = N
        self.fpga.register_map.K = K
        self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        C.invalidate()
        result = np.array(C).astype(np.float32) * (x_scale * self.w_scale)
        if self.bias is not None:
            result = result + self.bias
        return torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features)

sample_input = torch.randn(32, 768) * 0.5
layer0 = model.transformer.layer[0]
original_q_lin = layer0.attention.q_lin
fpga_q_lin = FPGAQuantizedLinear(original_q_lin, mmult_accel)

with torch.no_grad():
    out_cpu = original_q_lin(sample_input)
    out_fpga = fpga_q_lin(sample_input)

diff = (out_cpu - out_fpga).abs()
rel_error_norm = (out_cpu - out_fpga).norm() / out_cpu.norm() * 100
print("CPU output (5 gia tri dau):", out_cpu[0,:5])
print("FPGA output (5 gia tri dau):", out_fpga[0,:5])
print(f"Sai so tuong doi (norm): {rel_error_norm.item():.3f}%")

CPU output (5 gia tri dau): tensor([ 0.1930, -0.7376, -0.6320,  0.1769,  0.4602])
FPGA output (5 gia tri dau): tensor([ 0.1946, -0.7543, -0.6141,  0.1695,  0.4512])
Sai so tuong doi (norm): 3.346%


In [12]:
import copy

model_fpga = copy.deepcopy(model)
for layer in model_fpga.transformer.layer:
    layer.attention.q_lin = FPGAQuantizedLinear(layer.attention.q_lin, mmult_accel)
    layer.attention.k_lin = FPGAQuantizedLinear(layer.attention.k_lin, mmult_accel)
    layer.attention.v_lin = FPGAQuantizedLinear(layer.attention.v_lin, mmult_accel)
print("Da thay xong 18 lop Q/K/V.")

def run_test(sentence, max_length=None):
    if max_length:
        inputs = tokenizer(sentence, return_tensors="pt", padding="max_length", max_length=max_length, truncation=True)
    else:
        inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        t0 = time.time(); out_cpu = model(**inputs); t_cpu = time.time()-t0
        t0 = time.time(); out_fpga = model_fpga(**inputs); t_fpga = time.time()-t0
    hc, hf = out_cpu.last_hidden_state, out_fpga.last_hidden_state
    rel_err = (hc-hf).norm()/hc.norm()*100
    cos = torch.nn.functional.cosine_similarity(hc.flatten(), hf.flatten(), dim=0)
    n_tok = inputs['input_ids'].shape[1]
    print(f"So token={n_tok}  CPU={t_cpu*1000:.1f}ms  FPGA={t_fpga*1000:.1f}ms  SaiSo={rel_err.item():.2f}%  Cosine={cos.item():.4f}")

print("\n-- Cau ngan --")
run_test("The quick brown fox jumps over the lazy dog.")
print("\n-- Cau dai --")
run_test("The quick brown fox jumps over the lazy dog while the sun sets slowly behind the distant mountains, casting long shadows across the quiet valley below where a small river flows gently.")
print("\n-- N=64 (padding) --")
run_test("The quick brown fox jumps over the lazy dog.", max_length=64)

Da thay xong 18 lop Q/K/V.

-- Cau ngan --
So token=12  CPU=625.8ms  FPGA=468.0ms  SaiSo=11.58%  Cosine=0.9933

-- Cau dai --
So token=36  CPU=494.1ms  FPGA=750.0ms  SaiSo=7.69%  Cosine=0.9971

-- N=64 (padding) --
So token=64  CPU=676.2ms  FPGA=917.8ms  SaiSo=12.39%  Cosine=0.9923


In [13]:
from transformers import AutoModelForSequenceClassification
import torch.nn.functional as F

clf_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_model_name)
print(clf_model.config.id2label)

clf_model_fpga = copy.deepcopy(clf_model)
for layer in clf_model_fpga.distilbert.transformer.layer:
    layer.attention.q_lin = FPGAQuantizedLinear(layer.attention.q_lin, mmult_accel)
    layer.attention.k_lin = FPGAQuantizedLinear(layer.attention.k_lin, mmult_accel)
    layer.attention.v_lin = FPGAQuantizedLinear(layer.attention.v_lin, mmult_accel)

test_sentences = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "The service was terrible and I will never come back again.",
    "It was an okay experience, nothing special but not bad either.",
]

for sentence in test_sentences:
    inputs = clf_tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out_cpu = clf_model(**inputs)
        out_fpga = clf_model_fpga(**inputs)
    prob_cpu = F.softmax(out_cpu.logits, dim=-1)[0]
    prob_fpga = F.softmax(out_fpga.logits, dim=-1)[0]
    label_cpu = clf_model.config.id2label[prob_cpu.argmax().item()]
    label_fpga = clf_model.config.id2label[prob_fpga.argmax().item()]
    print(f"{sentence[:40]:42} CPU={label_cpu}({prob_cpu.max().item()*100:.1f}%)  FPGA={label_fpga}({prob_fpga.max().item()*100:.1f}%)")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{0: 'NEGATIVE', 1: 'POSITIVE'}
This movie was absolutely fantastic, I l   CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)
The service was terrible and I will neve   CPU=NEGATIVE(99.7%)  FPGA=NEGATIVE(99.7%)
It was an okay experience, nothing speci   CPU=POSITIVE(98.6%)  FPGA=POSITIVE(98.4%)


In [14]:
cfg = model.config
seq_len = 64
n_layers = cfg.n_layers
hidden = cfg.dim
ffn_dim = cfg.hidden_dim
n_heads = cfg.n_heads
d_k = hidden // n_heads

flops_qkv   = 3 * 2 * seq_len * hidden * hidden
flops_qkt   = 2 * n_heads * seq_len * seq_len * d_k
flops_av    = 2 * n_heads * seq_len * seq_len * d_k
flops_oproj = 2 * seq_len * hidden * hidden
flops_ffn   = 2 * seq_len * hidden * ffn_dim + 2 * seq_len * ffn_dim * hidden

flops_per_layer = flops_qkv + flops_qkt + flops_av + flops_oproj + flops_ffn
total_flops = flops_per_layer * n_layers
print(f"FLOPs moi tang: {flops_per_layer/1e6:.2f} MFLOPs")
print(f"Tong FLOPs ({n_layers} tang): {total_flops/1e9:.3f} GFLOPs")

sentence = "The quick brown fox jumps over the lazy dog."
inputs = tokenizer(sentence, return_tensors="pt", padding="max_length", max_length=seq_len, truncation=True)
with torch.no_grad():
    t0 = time.time(); out_cpu_full = model(**inputs); t_cpu_full = time.time()-t0
    t0 = time.time(); out_fpga_full = model_fpga(**inputs); t_fpga_full = time.time()-t0

print(f"\nCPU-only            : {t_cpu_full*1000:.2f} ms  ->  {total_flops/t_cpu_full/1e9:.3f} GFLOPs/s")
print(f"FPGA (Q/K/V offload): {t_fpga_full*1000:.2f} ms  ->  {total_flops/t_fpga_full/1e9:.3f} GFLOPs/s")

FLOPs moi tang: 918.55 MFLOPs
Tong FLOPs (6 tang): 5.511 GFLOPs

CPU-only            : 917.39 ms  ->  6.008 GFLOPs/s
FPGA (Q/K/V offload): 918.22 ms  ->  6.002 GFLOPs/s


In [15]:
class FPGAQKVAttention:
    def __init__(self, q_lin, k_lin, v_lin, fpga_ip, max_n=64):
        self.fpga = fpga_ip
        self.in_features = q_lin.in_features
        self.out_features = q_lin.out_features
        self.max_n = max_n
        self.linears = [q_lin, k_lin, v_lin]
        self.biases, self.w_scales, self.B_int8_list = [], [], []
        for lin in self.linears:
            W = lin.weight.detach().numpy()
            bias = lin.bias.detach().numpy().astype(np.float32) if lin.bias is not None else None
            w_scale = np.abs(W).max() / 127.0
            w_scale = w_scale if w_scale > 0 else 1e-8
            W_int8 = np.round(W / w_scale).astype(np.int8)
            self.biases.append(bias); self.w_scales.append(w_scale)
            self.B_int8_list.append(np.ascontiguousarray(W_int8.T))
        K, M = self.in_features, self.out_features
        self.A_buf = allocate(shape=(max_n, K), dtype=np.int8)
        self.B_buf = allocate(shape=(K, M), dtype=np.int8)
        self.C_buf = allocate(shape=(max_n, M), dtype=np.int32)
        self._cache = None

    def _run_one(self, X_int8, B_int8, N, K, M, update_A):
        self.A_buf[:N, :K] = X_int8
        self.B_buf[:K, :M] = B_int8
        if update_A: self.A_buf.flush()
        self.B_buf.flush()
        set_ptr('A_1','A_2', self.A_buf.physical_address)
        set_ptr('B_1','B_2', self.B_buf.physical_address)
        set_ptr('C_1','C_2', self.C_buf.physical_address)
        self.fpga.register_map.N = N; self.fpga.register_map.K = K; self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1 if update_A else 0
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        self.C_buf.invalidate()
        return np.array(self.C_buf[:N, :M])

    def compute_qkv(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        X_int8 = np.round(x2d / x_scale).astype(np.int8)
        outputs = []
        for i in range(3):
            C_int32 = self._run_one(X_int8, self.B_int8_list[i], N, K, M, update_A=(i==0))
            result = C_int32.astype(np.float32) * (x_scale * self.w_scales[i])
            if self.biases[i] is not None: result = result + self.biases[i]
            outputs.append(torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features))
        return outputs

class FPGAQKVWrapper(nn.Module):
    def __init__(self, shared, index):
        super().__init__()
        self.shared = shared; self.index = index
    def forward(self, x):
        if self.index == 0:
            self.shared._cache = self.shared.compute_qkv(x)
        return self.shared._cache[self.index]

model_fpga_opt = copy.deepcopy(model)
for layer in model_fpga_opt.transformer.layer:
    shared = FPGAQKVAttention(layer.attention.q_lin, layer.attention.k_lin, layer.attention.v_lin, mmult_accel, max_n=64)
    layer.attention.q_lin = FPGAQKVWrapper(shared, 0)
    layer.attention.k_lin = FPGAQKVWrapper(shared, 1)
    layer.attention.v_lin = FPGAQKVWrapper(shared, 2)
print("Da thay xong ban toi uu V1 (persistent-A).")

with torch.no_grad():
    t0 = time.time(); out_cpu = model(**inputs); t_cpu = time.time()-t0
    t0 = time.time(); out_fpga_opt = model_fpga_opt(**inputs); t_fpga_opt = time.time()-t0

hc, hf = out_cpu.last_hidden_state, out_fpga_opt.last_hidden_state
rel_err = (hc-hf).norm()/hc.norm()*100
cos = torch.nn.functional.cosine_similarity(hc.flatten(), hf.flatten(), dim=0)
print(f"CPU-only: {t_cpu*1000:.1f}ms   FPGA V1: {t_fpga_opt*1000:.1f}ms   SaiSo={rel_err.item():.2f}%   Cosine={cos.item():.4f}")

Da thay xong ban toi uu V1 (persistent-A).
CPU-only: 880.5ms   FPGA V1: 874.5ms   SaiSo=12.39%   Cosine=0.9923


In [16]:
class FPGAQKVAttentionV2:
    def __init__(self, q_lin, k_lin, v_lin, fpga_ip, max_n=64):
        self.fpga = fpga_ip
        self.in_features = q_lin.in_features
        self.out_features = q_lin.out_features
        self.max_n = max_n
        K, M = self.in_features, self.out_features
        self.A_buf = allocate(shape=(max_n, K), dtype=np.int8)
        self.C_buf = allocate(shape=(max_n, M), dtype=np.int32)
        self.biases, self.w_scales, self.B_bufs = [], [], []
        for lin in [q_lin, k_lin, v_lin]:
            W = lin.weight.detach().numpy()
            bias = lin.bias.detach().numpy().astype(np.float32) if lin.bias is not None else None
            w_scale = np.abs(W).max() / 127.0
            w_scale = w_scale if w_scale > 0 else 1e-8
            W_int8 = np.round(W / w_scale).astype(np.int8)
            B_buf = allocate(shape=(K, M), dtype=np.int8)
            B_buf[:] = np.ascontiguousarray(W_int8.T)
            B_buf.flush()
            self.biases.append(bias); self.w_scales.append(w_scale); self.B_bufs.append(B_buf)
        self._cache = None

    def _run_one(self, N, K, M, B_buf, update_A):
        if update_A: self.A_buf.flush()
        set_ptr('A_1','A_2', self.A_buf.physical_address)
        set_ptr('B_1','B_2', B_buf.physical_address)
        set_ptr('C_1','C_2', self.C_buf.physical_address)
        self.fpga.register_map.N = N; self.fpga.register_map.K = K; self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1 if update_A else 0
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        self.C_buf.invalidate()
        return np.array(self.C_buf[:N, :M])

    def compute_qkv(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        self.A_buf[:N, :K] = np.round(x2d / x_scale).astype(np.int8)
        outputs = []
        for i in range(3):
            C_int32 = self._run_one(N, K, M, self.B_bufs[i], update_A=(i==0))
            result = C_int32.astype(np.float32) * (x_scale * self.w_scales[i])
            if self.biases[i] is not None: result = result + self.biases[i]
            outputs.append(torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features))
        return outputs

model_fpga_opt2 = copy.deepcopy(model)
for layer in model_fpga_opt2.transformer.layer:
    shared2 = FPGAQKVAttentionV2(layer.attention.q_lin, layer.attention.k_lin, layer.attention.v_lin, mmult_accel, max_n=64)
    layer.attention.q_lin = FPGAQKVWrapper(shared2, 0)
    layer.attention.k_lin = FPGAQKVWrapper(shared2, 1)
    layer.attention.v_lin = FPGAQKVWrapper(shared2, 2)
print("Da thay xong ban toi uu V2.")

with torch.no_grad():
    t0 = time.time(); out_cpu2 = model(**inputs); t_cpu2 = time.time()-t0
    t0 = time.time(); out_fpga_v2 = model_fpga_opt2(**inputs); t_fpga_v2 = time.time()-t0

hc2, hf2 = out_cpu2.last_hidden_state, out_fpga_v2.last_hidden_state
rel_err2 = (hc2-hf2).norm()/hc2.norm()*100
cos2 = torch.nn.functional.cosine_similarity(hc2.flatten(), hf2.flatten(), dim=0)
print(f"CPU-only: {t_cpu2*1000:.1f}ms   FPGA V2: {t_fpga_v2*1000:.1f}ms   SaiSo={rel_err2.item():.2f}%   Cosine={cos2.item():.4f}")

Da thay xong ban toi uu V2.
CPU-only: 9212.3ms   FPGA V2: 1851.9ms   SaiSo=12.39%   Cosine=0.9923


In [1]:
import numpy as np
import torch
import torch.nn as nn
import time, copy
from pynq import Overlay, allocate

overlay = Overlay("mmult_accel.bit")
mmult_accel = overlay.mmult_accel_0

def set_ptr(reg1, reg2, addr):
    setattr(mmult_accel.register_map, reg1, addr & 0xFFFFFFFF)
    setattr(mmult_accel.register_map, reg2, (addr >> 32) & 0xFFFFFFFF)

print("Overlay + set_ptr san sang.")

Overlay + set_ptr san sang.


In [2]:
from transformers import AutoTokenizer, AutoModel

model = AutoModel.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sentence = "The quick brown fox jumps over the lazy dog."
inputs = tokenizer(sentence, return_tensors="pt", padding="max_length", max_length=64, truncation=True)
print("Model + inputs san sang.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model + inputs san sang.


In [3]:
class FPGAQKVAttention:
    def __init__(self, q_lin, k_lin, v_lin, fpga_ip, max_n=64):
        self.fpga = fpga_ip
        self.in_features = q_lin.in_features
        self.out_features = q_lin.out_features
        self.linears = [q_lin, k_lin, v_lin]
        self.biases, self.w_scales, self.B_int8_list = [], [], []
        for lin in self.linears:
            W = lin.weight.detach().numpy()
            bias = lin.bias.detach().numpy().astype(np.float32) if lin.bias is not None else None
            w_scale = np.abs(W).max() / 127.0
            w_scale = w_scale if w_scale > 0 else 1e-8
            W_int8 = np.round(W / w_scale).astype(np.int8)
            self.biases.append(bias); self.w_scales.append(w_scale)
            self.B_int8_list.append(np.ascontiguousarray(W_int8.T))
        K, M = self.in_features, self.out_features
        self.A_buf = allocate(shape=(max_n, K), dtype=np.int8)
        self.B_buf = allocate(shape=(K, M), dtype=np.int8)
        self.C_buf = allocate(shape=(max_n, M), dtype=np.int32)
        self._cache = None

    def _run_one(self, X_int8, B_int8, N, K, M, update_A):
        self.A_buf[:N, :K] = X_int8
        self.B_buf[:K, :M] = B_int8
        if update_A: self.A_buf.flush()
        self.B_buf.flush()
        set_ptr('A_1','A_2', self.A_buf.physical_address)
        set_ptr('B_1','B_2', self.B_buf.physical_address)
        set_ptr('C_1','C_2', self.C_buf.physical_address)
        self.fpga.register_map.N = N; self.fpga.register_map.K = K; self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1 if update_A else 0
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        self.C_buf.invalidate()
        return np.array(self.C_buf[:N, :M])

    def compute_qkv(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        X_int8 = np.round(x2d / x_scale).astype(np.int8)
        outputs = []
        for i in range(3):
            C_int32 = self._run_one(X_int8, self.B_int8_list[i], N, K, M, update_A=(i==0))
            result = C_int32.astype(np.float32) * (x_scale * self.w_scales[i])
            if self.biases[i] is not None: result = result + self.biases[i]
            outputs.append(torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features))
        return outputs

class FPGAQKVWrapper(nn.Module):
    def __init__(self, shared, index):
        super().__init__()
        self.shared = shared; self.index = index
    def forward(self, x):
        if self.index == 0:
            self.shared._cache = self.shared.compute_qkv(x)
        return self.shared._cache[self.index]

model_fpga_opt = copy.deepcopy(model)
for layer in model_fpga_opt.transformer.layer:
    shared = FPGAQKVAttention(layer.attention.q_lin, layer.attention.k_lin, layer.attention.v_lin, mmult_accel, max_n=64)
    layer.attention.q_lin = FPGAQKVWrapper(shared, 0)
    layer.attention.k_lin = FPGAQKVWrapper(shared, 1)
    layer.attention.v_lin = FPGAQKVWrapper(shared, 2)

with torch.no_grad():
    t0 = time.time(); out_cpu = model(**inputs); t_cpu = time.time()-t0
    t0 = time.time(); out_fpga_opt = model_fpga_opt(**inputs); t_fpga_opt = time.time()-t0

hc, hf = out_cpu.last_hidden_state, out_fpga_opt.last_hidden_state
rel_err = (hc-hf).norm()/hc.norm()*100
cos = torch.nn.functional.cosine_similarity(hc.flatten(), hf.flatten(), dim=0)
print(f"CPU-only: {t_cpu*1000:.1f}ms   FPGA V1: {t_fpga_opt*1000:.1f}ms   SaiSo={rel_err.item():.2f}%   Cosine={cos.item():.4f}")

CPU-only: 1299.9ms   FPGA V1: 850.6ms   SaiSo=12.39%   Cosine=0.9923


In [4]:
del model_fpga_opt  # giai phong RAM ban V1 truoc khi tao ban V2

class FPGAQKVAttentionV2:
    def __init__(self, q_lin, k_lin, v_lin, fpga_ip, max_n=64):
        self.fpga = fpga_ip
        self.in_features = q_lin.in_features
        self.out_features = q_lin.out_features
        K, M = self.in_features, self.out_features
        self.A_buf = allocate(shape=(max_n, K), dtype=np.int8)
        self.C_buf = allocate(shape=(max_n, M), dtype=np.int32)
        self.biases, self.w_scales, self.B_bufs = [], [], []
        for lin in [q_lin, k_lin, v_lin]:
            W = lin.weight.detach().numpy()
            bias = lin.bias.detach().numpy().astype(np.float32) if lin.bias is not None else None
            w_scale = np.abs(W).max() / 127.0
            w_scale = w_scale if w_scale > 0 else 1e-8
            W_int8 = np.round(W / w_scale).astype(np.int8)
            B_buf = allocate(shape=(K, M), dtype=np.int8)
            B_buf[:] = np.ascontiguousarray(W_int8.T)
            B_buf.flush()
            self.biases.append(bias); self.w_scales.append(w_scale); self.B_bufs.append(B_buf)
        self._cache = None

    def _run_one(self, N, K, M, B_buf, update_A):
        if update_A: self.A_buf.flush()
        set_ptr('A_1','A_2', self.A_buf.physical_address)
        set_ptr('B_1','B_2', B_buf.physical_address)
        set_ptr('C_1','C_2', self.C_buf.physical_address)
        self.fpga.register_map.N = N; self.fpga.register_map.K = K; self.fpga.register_map.M = M
        self.fpga.register_map.update_A = 1 if update_A else 0
        self.fpga.register_map.CTRL.AP_START = 1
        while self.fpga.register_map.CTRL.AP_DONE == 0:
            pass
        self.C_buf.invalidate()
        return np.array(self.C_buf[:N, :M])

    def compute_qkv(self, x):
        orig_shape = x.shape
        x2d = x.reshape(-1, self.in_features).detach().numpy()
        N, K = x2d.shape
        M = self.out_features
        x_scale = np.abs(x2d).max() / 127.0
        x_scale = x_scale if x_scale > 0 else 1e-8
        self.A_buf[:N, :K] = np.round(x2d / x_scale).astype(np.int8)
        outputs = []
        for i in range(3):
            C_int32 = self._run_one(N, K, M, self.B_bufs[i], update_A=(i==0))
            result = C_int32.astype(np.float32) * (x_scale * self.w_scales[i])
            if self.biases[i] is not None: result = result + self.biases[i]
            outputs.append(torch.from_numpy(result).reshape(*orig_shape[:-1], self.out_features))
        return outputs

model_fpga_opt2 = copy.deepcopy(model)
for layer in model_fpga_opt2.transformer.layer:
    shared2 = FPGAQKVAttentionV2(layer.attention.q_lin, layer.attention.k_lin, layer.attention.v_lin, mmult_accel, max_n=64)
    layer.attention.q_lin = FPGAQKVWrapper(shared2, 0)
    layer.attention.k_lin = FPGAQKVWrapper(shared2, 1)
    layer.attention.v_lin = FPGAQKVWrapper(shared2, 2)

with torch.no_grad():
    t0 = time.time(); out_cpu2 = model(**inputs); t_cpu2 = time.time()-t0
    t0 = time.time(); out_fpga_v2 = model_fpga_opt2(**inputs); t_fpga_v2 = time.time()-t0

hc2, hf2 = out_cpu2.last_hidden_state, out_fpga_v2.last_hidden_state
rel_err2 = (hc2-hf2).norm()/hc2.norm()*100
cos2 = torch.nn.functional.cosine_similarity(hc2.flatten(), hf2.flatten(), dim=0)
print(f"CPU-only: {t_cpu2*1000:.1f}ms   FPGA V2: {t_fpga_v2*1000:.1f}ms   SaiSo={rel_err2.item():.2f}%   Cosine={cos2.item():.4f}")

CPU-only: 897.2ms   FPGA V2: 832.7ms   SaiSo=12.39%   Cosine=0.9923


In [5]:
ITER_LONG = 200  # tang tu 20 len 200 vong lap
_, power_fpga_long, energy_fpga_total_long, t_fpga_total_long, n_fpga_long = measure_power_during(fpga_matmul_repeated, iterations=ITER_LONG)
energy_fpga_long = energy_fpga_total_long / ITER_LONG
t_fpga_long = t_fpga_total_long / ITER_LONG

print(f"FPGA (lap {ITER_LONG} lan): cong suat TB={power_fpga_long:.3f}W  thoi gian/lan={t_fpga_long*1000:.2f}ms  nang luong/lan={energy_fpga_long*1000:.2f}mJ  (so mau={n_fpga_long})")
print(f"So voi lan do 20 vong truoc: {power_fpga:.3f}W / {energy_fpga*1000:.2f}mJ")

NameError: name 'measure_power_during' is not defined

In [6]:
import threading

def read_power_watts():
    with open('/sys/class/hwmon/hwmon2/power1_input') as f:
        return int(f.read().strip()) / 1_000_000

def measure_power_during(func, *args, sample_interval=0.05, **kwargs):
    samples = []
    stop_flag = threading.Event()
    def sampler():
        while not stop_flag.is_set():
            samples.append(read_power_watts())
            time.sleep(sample_interval)
    t = threading.Thread(target=sampler)
    t.start()
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    stop_flag.set()
    t.join()
    avg_power = sum(samples) / len(samples) if samples else read_power_watts()
    energy_j = avg_power * elapsed
    return result, avg_power, energy_j, elapsed, len(samples)

N, K, M = 64, 768, 3072
A_np = np.random.randint(-5, 5, size=(N, K)).astype(np.int8)
B_np = np.random.randint(-5, 5, size=(K, M)).astype(np.int8)

def fpga_matmul_once():
    A = allocate(shape=(N, K), dtype=np.int8)
    B = allocate(shape=(K, M), dtype=np.int8)
    C = allocate(shape=(N, M), dtype=np.int32)
    A[:] = A_np; B[:] = B_np; C[:] = 0
    A.flush(); B.flush()
    set_ptr('A_1','A_2', A.physical_address)
    set_ptr('B_1','B_2', B.physical_address)
    set_ptr('C_1','C_2', C.physical_address)
    mmult_accel.register_map.N = N
    mmult_accel.register_map.K = K
    mmult_accel.register_map.M = M
    mmult_accel.register_map.update_A = 1
    mmult_accel.register_map.CTRL.AP_START = 1
    while mmult_accel.register_map.CTRL.AP_DONE == 0:
        pass
    C.invalidate()

def fpga_matmul_repeated(iterations):
    for _ in range(iterations):
        fpga_matmul_once()

ITER_LONG = 200
_, power_fpga_long, energy_fpga_total_long, t_fpga_total_long, n_fpga_long = measure_power_during(fpga_matmul_repeated, iterations=ITER_LONG)
energy_fpga_long = energy_fpga_total_long / ITER_LONG
t_fpga_long = t_fpga_total_long / ITER_LONG

print(f"FPGA (lap {ITER_LONG} lan): cong suat TB={power_fpga_long:.3f}W  thoi gian/lan={t_fpga_long*1000:.2f}ms  nang luong/lan={energy_fpga_long*1000:.2f}mJ  (so mau={n_fpga_long})")

FPGA (lap 200 lan): cong suat TB=3.901W  thoi gian/lan=71.16ms  nang luong/lan=277.57mJ  (so mau=201)


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

clf_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_model_name)

clf_model_fpga = copy.deepcopy(clf_model)
for layer in clf_model_fpga.distilbert.transformer.layer:
    shared = FPGAQKVAttentionV2(layer.attention.q_lin, layer.attention.k_lin, layer.attention.v_lin, mmult_accel, max_n=64)
    layer.attention.q_lin = FPGAQKVWrapper(shared, 0)
    layer.attention.k_lin = FPGAQKVWrapper(shared, 1)
    layer.attention.v_lin = FPGAQKVWrapper(shared, 2)

test_sentences = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "The service was terrible and I will never come back again.",
    "It was an okay experience, nothing special but not bad either.",
    "Best purchase I have made all year, highly recommend to everyone.",
    "Complete waste of money, do not buy this product.",
    "The food was delicious and the staff were very friendly.",
    "I am extremely disappointed with the quality of this item.",
    "A truly wonderful performance, the actors were brilliant.",
    "This is the worst customer service I have ever experienced.",
    "Pretty good overall, met my expectations for the price.",
    "The plot was boring and the pacing felt way too slow.",
    "Amazing product, works exactly as described, five stars.",
    "I regret buying this, it broke after just one use.",
    "The hotel room was clean, comfortable, and had a great view.",
    "Terrible experience from start to finish, avoid at all costs.",
    "Solid effort, though there is definitely room for improvement.",
]

agree = 0
conf_diffs = []
for sentence in test_sentences:
    inputs_s = clf_tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out_cpu_s = clf_model(**inputs_s)
        out_fpga_s = clf_model_fpga(**inputs_s)
    prob_cpu = F.softmax(out_cpu_s.logits, dim=-1)[0]
    prob_fpga = F.softmax(out_fpga_s.logits, dim=-1)[0]
    label_cpu = clf_model.config.id2label[prob_cpu.argmax().item()]
    label_fpga = clf_model.config.id2label[prob_fpga.argmax().item()]
    match = (label_cpu == label_fpga)
    agree += int(match)
    conf_diff = abs(prob_cpu.max().item() - prob_fpga.max().item()) * 100
    conf_diffs.append(conf_diff)
    print(f"{'OK ' if match else 'SAI'} CPU={label_cpu}({prob_cpu.max().item()*100:.1f}%)  FPGA={label_fpga}({prob_fpga.max().item()*100:.1f}%)  lech={conf_diff:.2f}%  | {sentence[:45]}")

print(f"\n=== Tong ket tren {len(test_sentences)} cau ===")
print(f"Ty le khop nhan: {agree}/{len(test_sentences)} = {agree/len(test_sentences)*100:.1f}%")
print(f"Do lech confidence trung binh: {sum(conf_diffs)/len(conf_diffs):.3f} diem %")
print(f"Do lech confidence lon nhat: {max(conf_diffs):.3f} diem %")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | This movie was absolutely fantastic, I loved 
OK  CPU=NEGATIVE(99.7%)  FPGA=NEGATIVE(99.7%)  lech=0.03%  | The service was terrible and I will never com
OK  CPU=POSITIVE(98.6%)  FPGA=POSITIVE(98.4%)  lech=0.18%  | It was an okay experience, nothing special bu
OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | Best purchase I have made all year, highly re
OK  CPU=NEGATIVE(100.0%)  FPGA=NEGATIVE(100.0%)  lech=0.00%  | Complete waste of money, do not buy this prod
OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | The food was delicious and the staff were ver
OK  CPU=NEGATIVE(100.0%)  FPGA=NEGATIVE(100.0%)  lech=0.00%  | I am extremely disappointed with the quality 
OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | A truly wonderful performance, the actors wer
OK  CPU=NEGATIVE(100.0%)  FPGA=NEGATIVE(100.0%)  lech=0.00%  | This is the worst customer service I have eve
OK  CPU=POSITIVE(100.0%

In [8]:
c_code = '''
#include <stdint.h>
void run_and_wait(volatile uint32_t *ctrl_reg) {
    uint32_t v = *ctrl_reg;
    *ctrl_reg = v | 0x1;      // bat AP_START
    while ((*ctrl_reg & 0x2) == 0) {
        // cho AP_DONE
    }
}
'''
with open('fpga_ctrl.c', 'w') as f:
    f.write(c_code)
print("Da ghi file fpga_ctrl.c")

Da ghi file fpga_ctrl.c


In [9]:
!gcc -O2 -shared -fPIC -o fpga_ctrl.so fpga_ctrl.c
!ls -la fpga_ctrl.so

-rwxr-xr-x 1 root root 7816 Jul 31 03:47 fpga_ctrl.so


In [10]:
import ctypes

lib = ctypes.CDLL('./fpga_ctrl.so')
lib.run_and_wait.argtypes = [ctypes.c_void_p]
lib.run_and_wait.restype = None

ctrl_addr = mmult_accel.mmio.array.ctypes.data
ctrl_ptr = ctypes.c_void_p(ctrl_addr)

print("Da nap thu vien C, dia chi thanh ghi CTRL:", hex(ctrl_addr))

Da nap thu vien C, dia chi thanh ghi CTRL: 0xffff8fdaa000


In [11]:
def fpga_matmul_once_C():
    A = allocate(shape=(N, K), dtype=np.int8)
    B = allocate(shape=(K, M), dtype=np.int8)
    C = allocate(shape=(N, M), dtype=np.int32)
    A[:] = A_np; B[:] = B_np; C[:] = 0
    A.flush(); B.flush()
    set_ptr('A_1','A_2', A.physical_address)
    set_ptr('B_1','B_2', B.physical_address)
    set_ptr('C_1','C_2', C.physical_address)
    mmult_accel.register_map.N = N
    mmult_accel.register_map.K = K
    mmult_accel.register_map.M = M
    mmult_accel.register_map.update_A = 1
    lib.run_and_wait(ctrl_ptr)
    C.invalidate()
    return np.array(C)

# Kiem tra dung truoc
C_expected = A_np.astype(np.int32) @ B_np.astype(np.int32)
C_result = fpga_matmul_once_C()
print("Dung:", np.array_equal(C_result, C_expected))

# So sanh toc do: Python polling vs C polling, lap nhieu lan
ITER_CMP = 50

t0 = time.time()
for _ in range(ITER_CMP):
    fpga_matmul_once()
t_python = (time.time() - t0) / ITER_CMP

t0 = time.time()
for _ in range(ITER_CMP):
    fpga_matmul_once_C()
t_c = (time.time() - t0) / ITER_CMP

print(f"Python polling: {t_python*1000:.3f} ms/lan")
print(f"C polling     : {t_c*1000:.3f} ms/lan")
print(f"Cai thien: {(t_python-t_c)/t_python*100:.1f}%")

Dung: True
Python polling: 71.946 ms/lan
C polling     : 77.470 ms/lan
Cai thien: -7.7%


In [12]:
import threading

def read_power_watts():
    with open('/sys/class/hwmon/hwmon2/power1_input') as f:
        return int(f.read().strip()) / 1_000_000

def measure_power_during(func, *args, sample_interval=0.05, **kwargs):
    samples = []
    stop_flag = threading.Event()
    def sampler():
        while not stop_flag.is_set():
            samples.append(read_power_watts())
            time.sleep(sample_interval)
    t = threading.Thread(target=sampler)
    t.start()
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    stop_flag.set()
    t.join()
    avg_power = sum(samples) / len(samples) if samples else read_power_watts()
    energy_j = avg_power * elapsed
    return result, avg_power, energy_j, elapsed, len(samples)

N, K, M = 64, 768, 3072
A_np = np.random.randint(-5, 5, size=(N, K)).astype(np.int8)
B_np = np.random.randint(-5, 5, size=(K, M)).astype(np.int8)

def fpga_matmul_once():
    A = allocate(shape=(N, K), dtype=np.int8)
    B = allocate(shape=(K, M), dtype=np.int8)
    C = allocate(shape=(N, M), dtype=np.int32)
    A[:] = A_np; B[:] = B_np; C[:] = 0
    A.flush(); B.flush()
    set_ptr('A_1','A_2', A.physical_address)
    set_ptr('B_1','B_2', B.physical_address)
    set_ptr('C_1','C_2', C.physical_address)
    mmult_accel.register_map.N = N
    mmult_accel.register_map.K = K
    mmult_accel.register_map.M = M
    mmult_accel.register_map.update_A = 1
    mmult_accel.register_map.CTRL.AP_START = 1
    while mmult_accel.register_map.CTRL.AP_DONE == 0:
        pass
    C.invalidate()

def fpga_matmul_repeated(iterations):
    for _ in range(iterations):
        fpga_matmul_once()

ITER_LONG = 200
_, power_fpga_long, energy_fpga_total_long, t_fpga_total_long, n_fpga_long = measure_power_during(fpga_matmul_repeated, iterations=ITER_LONG)
energy_fpga_long = energy_fpga_total_long / ITER_LONG
t_fpga_long = t_fpga_total_long / ITER_LONG

print(f"FPGA (lap {ITER_LONG} lan): cong suat TB={power_fpga_long:.3f}W  thoi gian/lan={t_fpga_long*1000:.2f}ms  nang luong/lan={energy_fpga_long*1000:.2f}mJ  (so mau={n_fpga_long})")

FPGA (lap 200 lan): cong suat TB=3.882W  thoi gian/lan=71.07ms  nang luong/lan=275.92mJ  (so mau=201)


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

clf_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_model_name)

clf_model_fpga = copy.deepcopy(clf_model)
for layer in clf_model_fpga.distilbert.transformer.layer:
    shared = FPGAQKVAttentionV2(layer.attention.q_lin, layer.attention.k_lin, layer.attention.v_lin, mmult_accel, max_n=64)
    layer.attention.q_lin = FPGAQKVWrapper(shared, 0)
    layer.attention.k_lin = FPGAQKVWrapper(shared, 1)
    layer.attention.v_lin = FPGAQKVWrapper(shared, 2)

test_sentences = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "The service was terrible and I will never come back again.",
    "It was an okay experience, nothing special but not bad either.",
    "Best purchase I have made all year, highly recommend to everyone.",
    "Complete waste of money, do not buy this product.",
    "The food was delicious and the staff were very friendly.",
    "I am extremely disappointed with the quality of this item.",
    "A truly wonderful performance, the actors were brilliant.",
    "This is the worst customer service I have ever experienced.",
    "Pretty good overall, met my expectations for the price.",
    "The plot was boring and the pacing felt way too slow.",
    "Amazing product, works exactly as described, five stars.",
    "I regret buying this, it broke after just one use.",
    "The hotel room was clean, comfortable, and had a great view.",
    "Terrible experience from start to finish, avoid at all costs.",
    "Solid effort, though there is definitely room for improvement.",
]

agree = 0
conf_diffs = []
for sentence in test_sentences:
    inputs_s = clf_tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out_cpu_s = clf_model(**inputs_s)
        out_fpga_s = clf_model_fpga(**inputs_s)
    prob_cpu = F.softmax(out_cpu_s.logits, dim=-1)[0]
    prob_fpga = F.softmax(out_fpga_s.logits, dim=-1)[0]
    label_cpu = clf_model.config.id2label[prob_cpu.argmax().item()]
    label_fpga = clf_model.config.id2label[prob_fpga.argmax().item()]
    match = (label_cpu == label_fpga)
    agree += int(match)
    conf_diff = abs(prob_cpu.max().item() - prob_fpga.max().item()) * 100
    conf_diffs.append(conf_diff)
    print(f"{'OK ' if match else 'SAI'} CPU={label_cpu}({prob_cpu.max().item()*100:.1f}%)  FPGA={label_fpga}({prob_fpga.max().item()*100:.1f}%)  lech={conf_diff:.2f}%  | {sentence[:45]}")

print(f"\n=== Tong ket tren {len(test_sentences)} cau ===")
print(f"Ty le khop nhan: {agree}/{len(test_sentences)} = {agree/len(test_sentences)*100:.1f}%")
print(f"Do lech confidence trung binh: {sum(conf_diffs)/len(conf_diffs):.3f} diem %")
print(f"Do lech confidence lon nhat: {max(conf_diffs):.3f} diem %")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | This movie was absolutely fantastic, I loved 
OK  CPU=NEGATIVE(99.7%)  FPGA=NEGATIVE(99.7%)  lech=0.03%  | The service was terrible and I will never com
OK  CPU=POSITIVE(98.6%)  FPGA=POSITIVE(98.4%)  lech=0.18%  | It was an okay experience, nothing special bu
OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | Best purchase I have made all year, highly re
OK  CPU=NEGATIVE(100.0%)  FPGA=NEGATIVE(100.0%)  lech=0.00%  | Complete waste of money, do not buy this prod
OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | The food was delicious and the staff were ver
OK  CPU=NEGATIVE(100.0%)  FPGA=NEGATIVE(100.0%)  lech=0.00%  | I am extremely disappointed with the quality 
OK  CPU=POSITIVE(100.0%)  FPGA=POSITIVE(100.0%)  lech=0.00%  | A truly wonderful performance, the actors wer
OK  CPU=NEGATIVE(100.0%)  FPGA=NEGATIVE(100.0%)  lech=0.00%  | This is the worst customer service I have eve
OK  CPU=POSITIVE(100.0%

In [14]:
c_code = '''
#include <stdint.h>
void run_and_wait(volatile uint32_t *ctrl_reg) {
    uint32_t v = *ctrl_reg;
    *ctrl_reg = v | 0x1;
    while ((*ctrl_reg & 0x2) == 0) {
    }
}
'''
with open('fpga_ctrl.c', 'w') as f:
    f.write(c_code)
print("Da ghi file fpga_ctrl.c")

Da ghi file fpga_ctrl.c


In [15]:
!gcc -O2 -shared -fPIC -o fpga_ctrl.so fpga_ctrl.c
!ls -la fpga_ctrl.so

-rwxr-xr-x 1 root root 7816 Jul 31 03:53 fpga_ctrl.so


In [16]:
import ctypes

lib = ctypes.CDLL('./fpga_ctrl.so')
lib.run_and_wait.argtypes = [ctypes.c_void_p]
lib.run_and_wait.restype = None

ctrl_addr = mmult_accel.mmio.array.ctypes.data
ctrl_ptr = ctypes.c_void_p(ctrl_addr)

print("Da nap thu vien C, dia chi thanh ghi CTRL:", hex(ctrl_addr))

Da nap thu vien C, dia chi thanh ghi CTRL: 0xffff8fdaa000


In [17]:
def fpga_matmul_once_C():
    A = allocate(shape=(N, K), dtype=np.int8)
    B = allocate(shape=(K, M), dtype=np.int8)
    C = allocate(shape=(N, M), dtype=np.int32)
    A[:] = A_np; B[:] = B_np; C[:] = 0
    A.flush(); B.flush()
    set_ptr('A_1','A_2', A.physical_address)
    set_ptr('B_1','B_2', B.physical_address)
    set_ptr('C_1','C_2', C.physical_address)
    mmult_accel.register_map.N = N
    mmult_accel.register_map.K = K
    mmult_accel.register_map.M = M
    mmult_accel.register_map.update_A = 1
    lib.run_and_wait(ctrl_ptr)
    C.invalidate()
    return np.array(C)

C_expected = A_np.astype(np.int32) @ B_np.astype(np.int32)
C_result = fpga_matmul_once_C()
print("Dung:", np.array_equal(C_result, C_expected))

ITER_CMP = 50

t0 = time.time()
for _ in range(ITER_CMP):
    fpga_matmul_once()
t_python = (time.time() - t0) / ITER_CMP

t0 = time.time()
for _ in range(ITER_CMP):
    fpga_matmul_once_C()
t_c = (time.time() - t0) / ITER_CMP

print(f"Python polling: {t_python*1000:.3f} ms/lan")
print(f"C polling     : {t_c*1000:.3f} ms/lan")
print(f"Cai thien: {(t_python-t_c)/t_python*100:.1f}%")

Dung: True
Python polling: 72.057 ms/lan
C polling     : 77.405 ms/lan
Cai thien: -7.4%
